<a href="https://colab.research.google.com/github/ilyass-hm-04/AGI-for-Financial-Trading/blob/salma_branch/data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install ta

In [ ]:
import yfinance as yf  #importer les données financier depuis yahoo finance
import pandas as pd
from sklearn.preprocessing import StandardScaler
import ta   #pour calculer les indicateurs d'analyse technique pour le trading (RSI,MACD....)
import os

In [ ]:
Train_data=["AAPL", "MSFT", "GOOGL", "AMZN", "TSLA"]
Test_data=["NVDA", "META"]     #pour généraliser en utlise un type de données different

In [ ]:
def download_data(tickers:list[str],start:str="2015-01-01",end:str="2023-12-31"):
  data={}
  for ticker in tickers:
    df=yf.download(ticker,start=start,end=end,progress=False)
    if df.empty:
      print(f"aucune donnée pour {ticker}")
      continue
    df.dropna(inplace=True)
    data[ticker]=df
  return data
#il retourne un dictionnaire ou chaque clé (ici ticker) à une dataframe


In [ ]:
def add_technical_indicators(df:pd.DataFrame):
   close = df["Close"].squeeze()    #avec .squeeze il transforme un dataframe en serie pour utiliser ta
   high  = df["High"].squeeze()
   low   = df["Low"].squeeze()
   vol   = df["Volume"].squeeze()

   df["RSI"]=ta.momentum.RSIIndicator(close=close,window=14).rsi()

   macd=ta.trend.MACD(close=close)
   df["MACD"]= macd.macd()
   df["MACD_signal"]= macd.macd_signal()
   df["MACD_diff"]= macd.macd_diff()

   bb = ta.volatility.BollingerBands(close=close, window=20, window_dev=2)
   df["BB_upper"] = bb.bollinger_hband()
   df["BB_lower"] = bb.bollinger_lband()
   df["BB_mid"]   = bb.bollinger_mavg()
   df["BB_width"] = (df["BB_upper"] - df["BB_lower"]) / df["BB_mid"]

   df["ATR"] = ta.volatility.AverageTrueRange(high=high, low=low, close=close, window=14).average_true_range()

   df["Return_1d"]  = close.pct_change(1)
   df["Return_5d"]  = close.pct_change(5)
   df["Return_20d"] = close.pct_change(20)

   df["Volume_norm"] = vol / vol.rolling(20).mean()

   df.dropna(inplace=True)
   return df

#cette fonction permet d'ajouter les indicateurs RSI (si le marché est trés acheté/vendu) MACD (détecte la tendence globale(monte/descent d'une maniére accélérer/ralentit))
#Bollinger Bands (indique les zones de prix et la volatilité) ATR (force des mouvement du marché) elle calcule aussi les returns (combien le prix varie)
#ce qui donne une dataset intelligentes pour ai


In [ ]:
def normalize_features(df: pd.DataFrame,scaler: StandardScaler | None = None,fit: bool = True):
  feature_cols = ["RSI", "MACD", "MACD_signal", "MACD_diff","BB_width", "ATR","Return_1d", "Return_5d", "Return_20d","Volume_norm"]
  if scaler is None:
    scaler=StandardScaler()
  df_feat = df[feature_cols].copy() #les colonnes utiles pour l’IA
  if fit:
        df_feat[feature_cols] = scaler.fit_transform(df_feat[feature_cols])
  else:
        df_feat[feature_cols] = scaler.transform(df_feat[feature_cols])

  return df_feat, scaler

In [ ]:
def prepare_all(tickers: list[str],start: str = "2015-01-01",end:str = "2023-12-31",scaler: StandardScaler | None = None,fit_scaler: bool = True):
    data = download_data(tickers, start, end)
    processed = {}

    for ticker, df in data.items():
        df = add_technical_indicators(df)
        processed[ticker] = df

    if fit_scaler:
        combined = pd.concat([
            processed[t][["RSI","MACD","MACD_signal","MACD_diff",
                           "BB_width","ATR",
                           "Return_1d","Return_5d","Return_20d","Volume_norm"]]
            for t in processed
        ])
        scaler = StandardScaler()
        scaler.fit(combined)

    normalized = {}
    for ticker, df in processed.items():
        df_feat, _ = normalize_features(df, scaler=scaler, fit=False)
        normalized[ticker] = df_feat

    return normalized, scaler

#pipline complet